In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 3.5.0 — gotowy


In [ ]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  

In [ ]:
from pyspark.sql.functions import col, window, avg

zad_1 = (
    df.filter(col("store") == "Gdańsk")                    
    .groupBy(window("timestamp", "1 hour"))                  
    .agg(avg("amount").alias("srednia_kwota"))             
    .orderBy("srednia_kwota")                              
)


zad_1.show(1, truncate=False)

In [ ]:
from pyspark.sql.functions import window, count

zad_2 = (
    df.groupBy(window("timestamp", "30 minutes"), "category") 
    .agg(count("tx_id").alias("liczba_tx"))                   
    .orderBy("window", "category")                            
)


(
    zad_2.filter(col("window.start").cast("string").like("%09:00:00%"))
    .show(truncate=False)
)

In [ ]:
from pyspark.sql.functions import window, count, desc

zad_3 = (
    df.groupBy(window("timestamp", "15 minutes"))            
    .agg(count("tx_id").alias("liczba_tx"))                  
    .orderBy(desc("liczba_tx")))                            


zad_3.show(1, truncate=False)